# 🌲 YOLOv8-seg 樹幹偵測訓練 — Leak-Free 版（Colab Pro）

**目標**：訓練一個**完全不含 Xiang 294 棵**的 trunk segmentation 模型，並用 Xiang 全集當 **external hold-out test**，得到誠實可寫進論文的數字。

---

## 為什麼要重練？
- 舊模型 `tree_trunk_seg_best.pt` 訓練集 80% 是 Xiang → benchmark 中所有 `yolomask` 配置都有 leakage，論文無法主張「在 Xiang 上達到 mAP 0.9」
- 新版 `merged_no_xiang/` dataset 完全排除 Xiang，留下 ~7,500+ 張（roboflow 多源 + kaggle urban_street）
- Xiang 294 棵變成**完全乾淨的 external test set**

## Colab Pro 配置建議
| GPU | 模型 | imgsz | batch | epochs | 預估時間 |
|-----|------|-------|-------|--------|---------|
| A100 40GB | yolov8m-seg | 960 | 32 | 100 | 60–90 min |
| A100 80GB / H100 | yolov8m-seg | 1024 | 48 | 100 | 40–60 min |
| L4 24GB | yolov8s-seg | 832 | 24 | 100 | 90–120 min |

> 推薦 **m-seg + 960**：精度/速度甜蜜點，mobile 不直接跑這版（手機跑 n-seg），m-seg 是 server-side。

---

## 執行順序
1. 連線到 GPU runtime（A100/H100）
2. 從上到下執行每個 cell
3. 訓練 → external eval (Xiang) → export → 下載 ZIP

## Step 0：確認 GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}')
    print(f'VRAM: {p.total_memory / 1024**3:.1f} GB')

## Step 1：安裝依賴

In [ ]:
!pip install -q ultralytics==8.3.* opencv-python-headless pyyaml
import ultralytics
ultralytics.checks()

## Step 2：從 Google Drive 載入 dataset

兩個 zip 已經上傳到你的 Drive：
- `MyDrive/tree_project_colab_upload/merged_no_xiang.zip` (~13.6 GB)
- `MyDrive/tree_project_colab_upload/xiang_external.zip` (~200 MB)

> 🔥 **重要**：開始跑這 cell 前，先確認本機 Drive 桌面版**已上傳完畢**（檔案旁邊綠勾✓）。
> 不然 Colab 會看到 0 byte 或不完整的 zip。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/tree_project_colab_upload'
import os
for f in ['merged_no_xiang.zip', 'xiang_external.zip']:
    p = f'{DRIVE_DIR}/{f}'
    assert os.path.exists(p), f'❌ 找不到 {p} — Drive 還沒同步完？'
    print(f'  ✓ {f}: {os.path.getsize(p)/1024/1024:.1f} MB')

In [ ]:
import os, shutil, time, zipfile
from pathlib import Path

DATA_DIR = Path('/content/data')

# Clean old partial/broken extracts first. This avoids stale files from earlier failed runs.
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

def safe_extract_zip(zip_path, out_dir):
    """Extract zip while normalizing Windows backslashes to Linux paths."""
    out_dir = Path(out_dir).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        print(f'  entries: {len(entries)}')
        for index, info in enumerate(entries, start=1):
            normalized_name = info.filename.replace('\\', '/').lstrip('/')
            if not normalized_name or '..' in Path(normalized_name).parts:
                continue
            target = (out_dir / normalized_name).resolve()
            if not str(target).startswith(str(out_dir)):
                continue
            if normalized_name.endswith('/') or info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst, length=1024 * 1024)
            if index % 2000 == 0:
                print(f'    extracted {index}/{len(entries)}')

for filename in ['merged_no_xiang.zip', 'xiang_external.zip']:
    src = f'{DRIVE_DIR}/{filename}'
    print(f'解壓 {filename} ({os.path.getsize(src)/1024/1024:.1f} MB) ...')
    t0 = time.time()
    safe_extract_zip(src, DATA_DIR)
    print(f'  完成 ({time.time()-t0:.0f}s)')

print('\n/content/data top-level:')
for p in sorted(DATA_DIR.iterdir()):
    print(' ', p)

print('\ndata.yaml files:')
for p in sorted(DATA_DIR.rglob('data.yaml')):
    print(' ', p)

## Step 3：修正 data.yaml 路徑（Colab 絕對路徑）

In [ ]:
import yaml, glob
from pathlib import Path

# 找出 merged_no_xiang 的根目錄
candidates = sorted(glob.glob('/content/data/**/data.yaml', recursive=True))
print('找到 data.yaml:', candidates)
if not candidates:
    print('\n/content/data 目錄檢查：')
    for p in sorted(Path('/content/data').glob('*')):
        print(' ', p)
    raise FileNotFoundError('找不到 data.yaml；請先重新執行 Step 2 的解壓 cell')

merged_candidates = [p for p in candidates if 'merged_no_xiang' in p]
yaml_path = Path(merged_candidates[0] if merged_candidates else candidates[0])
DATA_ROOT = str(yaml_path.parent)
print(f'DATA_ROOT = {DATA_ROOT}')

# 重寫 data.yaml 用絕對路徑
# 注意：實際結構是 train/images, train/labels（不是 images/train）
cfg = {
    'path': DATA_ROOT,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {0: 'tree_trunk'},
    'nc': 1,
}
with open(yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# 統計與 sanity check
for split in ['train', 'valid', 'test']:
    image_count = len(list(Path(DATA_ROOT, split, 'images').glob('*.*')))
    label_count = len(list(Path(DATA_ROOT, split, 'labels').glob('*.txt')))
    print(f'  {split}: {image_count} images / {label_count} labels')
    assert image_count > 0, f'{split}/images 是空的，解壓路徑不對'
    assert label_count > 0, f'{split}/labels 是空的，解壓路徑不對'

## Step 4：訓練（Leak-Free）

**關鍵設定**：
- `name='trunk_seg_no_xiang_v1'` — 不要覆蓋舊模型
- `cos_lr=True` + `optimizer='AdamW'` — 與舊版一致
- 增強參數針對戶外樹木調過
- `patience=20` — early stop

In [ ]:
from ultralytics import YOLO

# 自動依 GPU VRAM 決定 batch（模型固定 yolov8m-seg，與 server-side production 一致）
# 訓練 imgsz 維持 832 — 與 Vivobook iGPU 上線推論的 imgsz 對齊，避免 train/serve mismatch
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# server-side production 部署 imgsz 也會用 832（Vivobook OpenVINO iGPU 推論 ~95ms）
# 若你決定上線跑 imgsz=1024，把這裡也改 1024，但訓練/上線必須一致
BASE = 'yolov8m-seg.pt'
IMGSZ = 832

if vram_gb >= 70:        # A100 80GB / H100
    BATCH = 48
elif vram_gb >= 35:      # A100 40GB
    BATCH = 32
elif vram_gb >= 22:      # L4
    BATCH = 20
else:                    # T4 / 其他
    BATCH = 12

print(f'GPU VRAM {vram_gb:.0f}GB → {BASE} @ {IMGSZ}px, batch={BATCH}')
print(f'(server production target: yolov8m-seg @ {IMGSZ}px on Intel Arc iGPU via OpenVINO FP16)')

model = YOLO(BASE)

results = model.train(
    data=f'{DATA_ROOT}/data.yaml',
    epochs=100,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    workers=8,
    patience=20,
    save=True,
    save_period=20,
    name='trunk_seg_no_xiang_v1',
    project='/content/runs',
    exist_ok=True,

    # 優化器
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,

    # 增強（戶外樹木）
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.3, shear=2.0,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.0,
    close_mosaic=10,

    # mask
    overlap_mask=True, mask_ratio=4,
    box=7.5, cls=0.5, dfl=1.5,
)

BEST_PT = '/content/runs/trunk_seg_no_xiang_v1/weights/best.pt'
print(f'\n✅ 訓練完成 → {BEST_PT}')

## Step 5：In-Domain 評估（merged_no_xiang test split）

這是「同分布測試」，會跟訓練資料同源（roboflow + urban_street）。
數字會比 Xiang 外部測試**好看**——這是預期的。

In [ ]:
from ultralytics import YOLO

best = YOLO(BEST_PT)

print('=== In-Domain Test (merged_no_xiang/test) ===')
in_metrics = best.val(
    data=f'{DATA_ROOT}/data.yaml',
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    save_json=True,
    name='val_in_domain',
    project='/content/runs',
)
print(f'\nbox  mAP50:    {in_metrics.box.map50:.4f}')
print(f'box  mAP50-95: {in_metrics.box.map:.4f}')
print(f'mask mAP50:    {in_metrics.seg.map50:.4f}')
print(f'mask mAP50-95: {in_metrics.seg.map:.4f}')

## Step 6：⭐ External Hold-Out 評估（Xiang 294 棵）

**這是論文的關鍵數字**——模型從未見過任何一張 Xiang 圖片。

需要先把 Xiang 的 RGB+Seg mask 轉成 YOLO segmentation 格式。

In [ ]:
import os, glob, cv2, yaml, numpy as np
from pathlib import Path

# Xiang 解壓後的目錄結構：tree/treeRGB/*.jpg + tree/treeSeg/*-tm.jpg
XIANG_ROOT = None
for cand in glob.glob('/content/data/**/treeRGB', recursive=True):
    XIANG_ROOT = str(Path(cand).parent)
    break
assert XIANG_ROOT, '找不到 Xiang treeRGB 資料夾'
print(f'Xiang root: {XIANG_ROOT}')

XIANG_OUT = '/content/xiang_test'
os.makedirs(f'{XIANG_OUT}/images/test', exist_ok=True)
os.makedirs(f'{XIANG_OUT}/labels/test', exist_ok=True)

rgb_files = sorted(glob.glob(f'{XIANG_ROOT}/treeRGB/*'))
print(f'共 {len(rgb_files)} 張 RGB')

def find_xiang_seg(rgb_path):
    """Xiang RGB: rgb-xxx.jpg; mask: rgb-xxx-tm.jpg."""
    stem = Path(rgb_path).stem
    candidates = [
        f'{XIANG_ROOT}/treeSeg/{stem}-tm.jpg',
        f'{XIANG_ROOT}/treeSeg/{stem}-tm.png',
        f'{XIANG_ROOT}/treeSeg/{stem}.jpg',
        f'{XIANG_ROOT}/treeSeg/{stem}.png',
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return None

def mask_to_yolo_polys(mask_path, img_w, img_h):
    """二值 mask → YOLO seg polygon（normalized）。"""
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return []
    if m.shape != (img_h, img_w):
        m = cv2.resize(m, (img_w, img_h), interpolation=cv2.INTER_NEAREST)
    _, binm = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_KCOS)
    polys = []
    for c in contours:
        if cv2.contourArea(c) < 200:
            continue
        c = c.reshape(-1, 2).astype(np.float32)
        c[:, 0] /= img_w
        c[:, 1] /= img_h
        if len(c) < 3:
            continue
        polys.append(c.flatten().tolist())
    return polys

n_ok, n_missing_seg, n_bad_img, n_empty_mask = 0, 0, 0, 0
missing_examples = []
empty_examples = []

for rgb in rgb_files:
    stem = Path(rgb).stem
    seg = find_xiang_seg(rgb)
    if seg is None:
        n_missing_seg += 1
        if len(missing_examples) < 5:
            missing_examples.append(Path(rgb).name)
        continue

    img = cv2.imread(rgb)
    if img is None:
        n_bad_img += 1
        continue

    h, w = img.shape[:2]
    polys = mask_to_yolo_polys(seg, w, h)
    if not polys:
        n_empty_mask += 1
        if len(empty_examples) < 5:
            empty_examples.append(Path(seg).name)
        continue

    dst_img = f'{XIANG_OUT}/images/test/{stem}.jpg'
    cv2.imwrite(dst_img, img, [cv2.IMWRITE_JPEG_QUALITY, 95])

    with open(f'{XIANG_OUT}/labels/test/{stem}.txt', 'w') as f:
        for p in polys:
            f.write('0 ' + ' '.join(f'{v:.6f}' for v in p) + '\n')
    n_ok += 1

print(f'轉換完成：{n_ok} ok / {n_missing_seg} missing_seg / {n_bad_img} bad_img / {n_empty_mask} empty_mask')
if missing_examples:
    print('missing_seg examples:', missing_examples)
if empty_examples:
    print('empty_mask examples:', empty_examples)
assert n_ok > 0, 'Xiang 轉換仍為 0，請貼出上方 missing/empty examples'

with open(f'{XIANG_OUT}/data.yaml', 'w') as f:
    yaml.safe_dump({
        'path': XIANG_OUT,
        'train': 'images/test',
        'val': 'images/test',
        'test': 'images/test',
        'names': {0: 'tree_trunk'},
        'nc': 1,
    }, f, sort_keys=False)

print(f'Xiang YOLO dataset ready: {XIANG_OUT}')

In [ ]:
print('=== ⭐ External Hold-Out (Xiang 294) ===')
ext_metrics = best.val(
    data=f'{XIANG_OUT}/data.yaml',
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    save_json=True,
    name='val_xiang_external',
    project='/content/runs',
)
print(f'\nbox  mAP50:    {ext_metrics.box.map50:.4f}')
print(f'box  mAP50-95: {ext_metrics.box.map:.4f}')
print(f'mask mAP50:    {ext_metrics.seg.map50:.4f}')
print(f'mask mAP50-95: {ext_metrics.seg.map:.4f}')

# 把兩組數字存成 CSV 給論文用
import json
summary = {
    'in_domain': {
        'box_map50': float(in_metrics.box.map50),
        'box_map':   float(in_metrics.box.map),
        'mask_map50': float(in_metrics.seg.map50),
        'mask_map':   float(in_metrics.seg.map),
    },
    'xiang_external': {
        'box_map50': float(ext_metrics.box.map50),
        'box_map':   float(ext_metrics.box.map),
        'mask_map50': float(ext_metrics.seg.map50),
        'mask_map':   float(ext_metrics.seg.map),
    },
}
with open('/content/runs/eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\n=== Summary ===')
print(json.dumps(summary, indent=2))

## Step 7：Export 給 server (ONNX + OpenVINO) 與手機 (TFLite)

> server 端 (Vivobook NPU) 跑 OpenVINO IR；手機備援跑 TFLite INT8（如果是 m-seg 太大就改 export n-seg 的小版本）。

In [ ]:
# ONNX
best.export(format='onnx', imgsz=IMGSZ, opset=13, dynamic=False, simplify=True)
print('✓ ONNX')

# OpenVINO FP16 — 直接 swap 到 Vivobook 的 ml_service/trunk_detector_training/tree_trunk_seg_best_openvino_model/
best.export(format='openvino', imgsz=IMGSZ, half=True)
print(f'✓ OpenVINO FP16 @ {IMGSZ}px (Vivobook iGPU 推論用)')

# TFLite INT8 對這次 m-seg server 模型不是必要產物，且 Colab/TensorFlow conversion 常卡很久。
# 手機端之後會另訓 yolov8n-detect；不要用這顆 m-seg 當 mobile production。
EXPORT_TFLITE = False
if EXPORT_TFLITE:
    try:
        best.export(format='tflite', imgsz=640, int8=True, data=f'{DATA_ROOT}/data.yaml')
        print('✓ TFLite INT8 @ 640px (optional mobile fallback)')
    except Exception as e:
        print(f'⚠️ TFLite 失敗（可跳過）：{e}')
else:
    print('↷ Skip TFLite export (not needed for server OpenVINO deployment)')

## Step 8：打包下載

In [ ]:
import os, glob, shutil
from pathlib import Path
from google.colab import files

WEIGHTS_DIR = '/content/runs/trunk_seg_no_xiang_v1/weights'
EXPORT_ROOT = '/content/export_no_xiang_v1'

if os.path.exists(EXPORT_ROOT):
    shutil.rmtree(EXPORT_ROOT)
os.makedirs(EXPORT_ROOT, exist_ok=True)

def copy_any(src, dst_dir):
    src_path = Path(src)
    dst = Path(dst_dir) / src_path.name
    if src_path.is_dir():
        shutil.copytree(src_path, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst)

# 複製 weights 內所有重要產物：best.pt / best.onnx / best_openvino_model / optional tflite
for src in glob.glob(f'{WEIGHTS_DIR}/*'):
    copy_any(src, EXPORT_ROOT)

# 評估摘要
summary_path = '/content/runs/eval_summary.json'
if os.path.exists(summary_path):
    shutil.copy2(summary_path, EXPORT_ROOT)
else:
    print('⚠️ 找不到 eval_summary.json；若你剛剛只跑到 external eval，先重新跑 summary cell')

# 各 val 的結果資料夾；Ultralytics 若重跑會自動產生 val_xiang_external2/3，所以用 glob 抓
for pattern in ['val_in_domain*', 'val_xiang_external*']:
    for src in sorted(glob.glob(f'/content/runs/{pattern}')):
        copy_any(src, EXPORT_ROOT)

# 訓練曲線
train_dir = '/content/runs/trunk_seg_no_xiang_v1'
for filename in ['results.csv', 'results.png', 'confusion_matrix.png', 'args.yaml']:
    src = f'{train_dir}/{filename}'
    if os.path.exists(src):
        shutil.copy2(src, EXPORT_ROOT)

# 打包
zip_base = '/content/trunk_seg_no_xiang_v1'
zip_path = f'{zip_base}.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_base, 'zip', EXPORT_ROOT)
print(f'✓ 已打包 {zip_path}')
print(f'  大小：{os.path.getsize(zip_path)/1024/1024:.1f} MB')
files.download(zip_path)

## Step 9（可選）：Drive 備份

避免 Colab session 結束遺失。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, datetime
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst_dir = f'/content/drive/MyDrive/tree_project/yolo_runs/{ts}_no_xiang_v1'
os.makedirs(dst_dir, exist_ok=True)
shutil.copytree(EXPORT_ROOT, f'{dst_dir}/export', dirs_exist_ok=True)
shutil.copy('/content/trunk_seg_no_xiang_v1.zip', dst_dir)
print(f'✓ 備份到 {dst_dir}')

---

## 📋 訓練後本機要做的事

1. **解壓** `trunk_seg_no_xiang_v1.zip` 到 `c:\projects\tree_project\project_code\backend\ml_service\trunk_detector_training\runs\no_xiang_v1\`
2. **保留舊模型** `tree_trunk_seg_best.pt` 不要刪 — 論文要對比 leak vs no-leak
3. **替換 server YOLO**：把新的 OpenVINO IR 部署到 Vivobook（先測 `use_server_yolo_mask=true`，YOLO endpoint 應該還是回相同 schema）
4. **重跑 31 棵 benchmark Config B (refdist)**：這時 `yolomask` 配置就是**真正乾淨**的數字了
5. **論文 §結果 表 X**：寫 3 行
   - in-domain (merged_no_xiang test): mAP50 = ?
   - **external (Xiang 294, leak-free): mAP50 = ?** ← 這個數字最重要
   - field test (your 31 trees, DBH MAE)：4.22 cm @ 1.0–1.5m bin / 校準後 2.92 cm

## ⚠️ 常見坑
- Colab Pro 12 小時上限：100 epochs m-seg 不會碰到，但 200+ epochs 要切兩段（用 `resume`）
- A100 偶爾 OOM：把 batch 降一半，或用 `cache='ram'` 改成 `cache=False`
- Xiang mask 是二值 PNG，但有些可能是 RGB → 轉灰階已處理
- 如果 `mask polygon` 在 val 過程出錯，檢查 contour 點數是否 ≥3（已過濾）